# ARC-v0.16 — Threshold Sensitivity and Alpha-Controlled Robustness

This notebook performs the two remaining robustness analyses for the SIGIR Full Paper:

1. threshold sensitivity for \(\epsilon \in \{0, 0.001, 0.002, 0.005, 0.01\}\);
2. within-\(\alpha\) and alpha-centered FIT↔validation robustness.

It also reports signed-harm sensitivity for thresholds \(\epsilon \ge 0.002\), where ARC-v0.15.1 provides full signed coverage.

The notebook reads existing ARC-v0.13 and ARC-v0.15.1 artifacts only. It does not rerun the full FEVER retrieval sweep and does not access test data.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, warnings
import numpy as np
import pandas as pd
from google.colab import drive
from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import LinearRegression

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 20260816
THRESHOLDS = [0.0, 0.001, 0.002, 0.005, 0.01]

DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.is_dir():
    drive.mount("/content/drive")

ARC_ROOT = DRIVE_ROOT / "rag-pq-checkpoints" / "arc-v0"
V013_ROOT = ARC_ROOT / "fever-boundary-external-replication-v013"
PREFERRED_V013 = V013_ROOT / "20260817-140640"

def is_complete_v013(path):
    return (
        path.is_dir()
        and len(list(path.glob("fit-*.parquet"))) == 44
        and len(list(path.glob("validation-*.parquet"))) == 44
        and (path / "v013_fever_boundary_protocol.json").is_file()
        and (path / "v013_validation_continuation_report.json").is_file()
    )

if is_complete_v013(PREFERRED_V013):
    V013_RUN = PREFERRED_V013
else:
    candidates = sorted(
        [p for p in V013_ROOT.iterdir() if p.is_dir() and is_complete_v013(p)],
        reverse=True,
    )
    assert candidates, "No complete ARC-v0.13 run found"
    V013_RUN = candidates[0]

V016_ROOT = ARC_ROOT / "threshold-alpha-robustness-v016"
V016_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
OUT = V016_ROOT / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)

def sha256_file(path, chunk_size=16*1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

print("v0.13 source:", V013_RUN)
print("Thresholds:", THRESHOLDS)
print("Output:", OUT)
print("ARC-v0.16 PREFLIGHT — PASS")


In [ ]:
PROTOCOL_PATH = V013_RUN / "v013_fever_boundary_protocol.json"
VALIDATION_REPORT_PATH = V013_RUN / "v013_validation_continuation_report.json"

protocol = json.loads(PROTOCOL_PATH.read_text(encoding="utf-8"))
validation_report = json.loads(VALIDATION_REPORT_PATH.read_text(encoding="utf-8"))

assert protocol["status"] == "FEVER_BOUNDARY_EXTERNAL_REPLICATION_SEALED_BEFORE_SWEEP"
assert validation_report["status"] == "ARC_V013_FEVER_VALIDATION_CONTINUATION_COMPLETE"
assert protocol["test_access_allowed"] is False
assert validation_report["test_accessed"] is False

print("Protocol SHA:", sha256_file(PROTOCOL_PATH))
print("Validation report SHA:", sha256_file(VALIDATION_REPORT_PATH))
print("Frozen primary epsilon:", protocol["regime_threshold_abs_slope"])
print("V0.13 SOURCE INTEGRITY — PASS")


In [ ]:
fit_files = sorted(V013_RUN.glob("fit-*.parquet"))
val_files = sorted(V013_RUN.glob("validation-*.parquet"))

assert len(fit_files) == 44
assert len(val_files) == 44

fit_traj = pd.concat([pd.read_parquet(p) for p in fit_files], ignore_index=True)
val_traj = pd.concat([pd.read_parquet(p) for p in val_files], ignore_index=True)

assert fit_traj["config_key"].nunique() == 44
assert val_traj["config_key"].nunique() == 44
assert fit_traj["query_id"].nunique() == 3350
assert val_traj["query_id"].nunique() == 3316

print("FIT trajectory:", fit_traj.shape)
print("VAL trajectory:", val_traj.shape)
print("TRAJECTORY LOAD — PASS")


In [ ]:
GROUP_COLS = [
    "query_id","low","high","method","alpha","k","temperature","config_key"
]

def reconstruct_h3(df):
    rows = []
    for keys, g in df.groupby(GROUP_COLS, dropna=False, sort=False):
        g = g.sort_values("iteration")
        x = g["iteration"].to_numpy(np.float64)
        y = g["abs_utility_gap"].to_numpy(np.float64)
        row = dict(zip(GROUP_COLS, keys))
        row["H3_abs_slope"] = float(np.polyfit(x, y, 1)[0])
        rows.append(row)
    return pd.DataFrame(rows)

fit = reconstruct_h3(fit_traj)
val = reconstruct_h3(val_traj)

assert len(fit) == 3350 * 44
assert len(val) == 3316 * 44

print("FIT slopes:", fit.shape)
print("VAL slopes:", val.shape)
print("H3 RECONSTRUCTION — PASS")


In [ ]:
def regime_summary(df, split_name, eps):
    h = df["H3_abs_slope"].to_numpy(np.float64)
    return {
        "split": split_name,
        "epsilon": float(eps),
        "observations": int(len(df)),
        "amplifying_fraction": float((h > eps).mean()),
        "stable_or_null_fraction": float((np.abs(h) <= eps).mean()),
        "reversal_fraction": float((h < -eps).mean()),
        "exact_zero_fraction": float((h == 0).mean()),
    }

regime_sensitivity = pd.DataFrame([
    regime_summary(df, split_name, eps)
    for eps in THRESHOLDS
    for split_name, df in [("fit", fit), ("validation", val)]
])

display(regime_sensitivity)
regime_sensitivity.to_csv(OUT / "v016_threshold_regime_sensitivity.csv", index=False)
print("THRESHOLD REGIME SENSITIVITY — COMPLETE")


In [ ]:
def config_risk(df, eps):
    return (
        df.assign(is_amp=(df["H3_abs_slope"] > eps).astype(float))
        .groupby(
            ["config_key","method","alpha","k","temperature"],
            dropna=False,
            as_index=False,
        )
        .agg(amplification_fraction=("is_amp","mean"))
    )

config_tables = {}
corr_rows = []

for eps in THRESHOLDS:
    f = config_risk(fit, eps)
    v = config_risk(val, eps)

    joined = f.merge(
        v,
        on=["config_key","method","alpha","k","temperature"],
        how="inner",
        suffixes=("_fit","_val"),
        validate="one_to_one",
    )
    assert len(joined) == 44
    config_tables[eps] = joined

    x = joined["amplification_fraction_fit"].to_numpy(np.float64)
    y = joined["amplification_fraction_val"].to_numpy(np.float64)

    pr = pearsonr(x, y)
    sr = spearmanr(x, y)

    corr_rows.append({
        "epsilon": float(eps),
        "config_count": 44,
        "pearson_r": float(pr.statistic),
        "pearson_p": float(pr.pvalue),
        "spearman_rho": float(sr.statistic),
        "spearman_p": float(sr.pvalue),
    })

config_corr_sensitivity = pd.DataFrame(corr_rows)
display(config_corr_sensitivity)
config_corr_sensitivity.to_csv(
    OUT / "v016_threshold_config_fit_val_correlations.csv",
    index=False,
)
print("CONFIG-LEVEL THRESHOLD ROBUSTNESS — COMPLETE")


In [ ]:
dose_rows = []

for eps in THRESHOLDS:
    for split_name, df in [("fit", fit), ("validation", val)]:
        tmp = (
            df.assign(is_amp=(df["H3_abs_slope"] > eps).astype(float))
            .groupby("alpha", as_index=False)
            .agg(amplification_fraction=("is_amp","mean"))
        )
        tmp["split"] = split_name
        tmp["epsilon"] = float(eps)
        dose_rows.append(tmp)

dose_sensitivity = pd.concat(dose_rows, ignore_index=True)[
    ["epsilon","split","alpha","amplification_fraction"]
]
display(dose_sensitivity)

mono_rows = []
for eps in THRESHOLDS:
    for split_name in ["fit","validation"]:
        g = dose_sensitivity[
            (dose_sensitivity["epsilon"] == eps)
            & (dose_sensitivity["split"] == split_name)
        ].sort_values("alpha")
        vals = g["amplification_fraction"].to_numpy(np.float64)
        diffs = np.diff(vals)
        mono_rows.append({
            "epsilon": float(eps),
            "split": split_name,
            "monotone_non_decreasing": bool(np.all(diffs >= -1e-15)),
            "strictly_increasing": bool(np.all(diffs > 0)),
            "minimum_step": float(diffs.min()),
        })

monotonicity = pd.DataFrame(mono_rows)
display(monotonicity)

dose_sensitivity.to_csv(
    OUT / "v016_threshold_alpha_dose_response.csv",
    index=False,
)
monotonicity.to_csv(
    OUT / "v016_threshold_alpha_monotonicity.csv",
    index=False,
)
print("ALPHA DOSE-RESPONSE SENSITIVITY — COMPLETE")


In [ ]:
within_alpha_rows = []

for eps in THRESHOLDS:
    joined = config_tables[eps]

    for alpha, g in joined.groupby("alpha", sort=True):
        x = g["amplification_fraction_fit"].to_numpy(np.float64)
        y = g["amplification_fraction_val"].to_numpy(np.float64)

        if np.std(x) <= 1e-15 or np.std(y) <= 1e-15:
            pearson_r = np.nan
            pearson_p = np.nan
        else:
            pr = pearsonr(x, y)
            pearson_r = float(pr.statistic)
            pearson_p = float(pr.pvalue)

        sr = spearmanr(x, y)

        within_alpha_rows.append({
            "epsilon": float(eps),
            "alpha": float(alpha),
            "config_count": int(len(g)),
            "pearson_r": pearson_r,
            "pearson_p": pearson_p,
            "spearman_rho": float(sr.statistic) if np.isfinite(sr.statistic) else np.nan,
            "spearman_p": float(sr.pvalue) if np.isfinite(sr.pvalue) else np.nan,
        })

within_alpha_corr = pd.DataFrame(within_alpha_rows)
display(within_alpha_corr)
within_alpha_corr.to_csv(
    OUT / "v016_within_alpha_fit_val_correlations.csv",
    index=False,
)
print("WITHIN-ALPHA CORRELATIONS — COMPLETE")


In [ ]:
resid_rows = []
resid_detail = []

for eps in THRESHOLDS:
    joined = config_tables[eps].copy()

    joined["fit_alpha_mean"] = (
        joined.groupby("alpha")["amplification_fraction_fit"].transform("mean")
    )
    joined["val_alpha_mean"] = (
        joined.groupby("alpha")["amplification_fraction_val"].transform("mean")
    )

    joined["fit_within_alpha_residual"] = (
        joined["amplification_fraction_fit"] - joined["fit_alpha_mean"]
    )
    joined["val_within_alpha_residual"] = (
        joined["amplification_fraction_val"] - joined["val_alpha_mean"]
    )

    x = joined["fit_within_alpha_residual"].to_numpy(np.float64)
    y = joined["val_within_alpha_residual"].to_numpy(np.float64)

    pr = pearsonr(x, y)
    sr = spearmanr(x, y)

    resid_rows.append({
        "epsilon": float(eps),
        "config_count": int(len(joined)),
        "within_alpha_centered_pearson_r": float(pr.statistic),
        "within_alpha_centered_pearson_p": float(pr.pvalue),
        "within_alpha_centered_spearman_rho": float(sr.statistic),
        "within_alpha_centered_spearman_p": float(sr.pvalue),
    })

    joined["epsilon"] = float(eps)
    resid_detail.append(joined)

residual_corr = pd.DataFrame(resid_rows)
display(residual_corr)

residual_corr.to_csv(
    OUT / "v016_alpha_centered_residual_correlations.csv",
    index=False,
)
pd.concat(resid_detail, ignore_index=True).to_csv(
    OUT / "v016_alpha_centered_config_detail.csv",
    index=False,
)
print("ALPHA-CENTERED RESIDUAL ROBUSTNESS — COMPLETE")


In [ ]:
reg_rows = []

for eps in THRESHOLDS:
    for split_name, df in [("fit", fit), ("validation", val)]:
        cfg = config_risk(df, eps).copy()
        cfg["is_softmax"] = (cfg["method"] == "softmax").astype(float)
        cfg["log_k"] = np.log(cfg["k"].astype(float))
        cfg["temperature_numeric"] = (
            pd.to_numeric(cfg["temperature"], errors="coerce").fillna(1.0)
        )

        feature_cols = ["alpha","log_k","is_softmax","temperature_numeric"]
        X = cfg[feature_cols].to_numpy(np.float64)
        y = cfg["amplification_fraction"].to_numpy(np.float64)

        model = LinearRegression().fit(X, y)

        reg_rows.append({
            "epsilon": float(eps),
            "split": split_name,
            "intercept": float(model.intercept_),
            "beta_alpha": float(model.coef_[0]),
            "beta_log_k": float(model.coef_[1]),
            "beta_is_softmax": float(model.coef_[2]),
            "beta_temperature": float(model.coef_[3]),
            "r2": float(model.score(X, y)),
        })

regression_audit = pd.DataFrame(reg_rows)
display(regression_audit)
regression_audit.to_csv(
    OUT / "v016_config_regression_consistency.csv",
    index=False,
)
print("CONFIG REGRESSION CONSISTENCY — COMPLETE")


In [ ]:
V0151_ROOT = ARC_ROOT / "signed-harm-full-coverage-repair-v0151"

signed_candidates = []
if V0151_ROOT.is_dir():
    for run in sorted(
        [p for p in V0151_ROOT.iterdir() if p.is_dir()],
        reverse=True,
    ):
        candidate = run / "v0151_full_coverage_signed_amplification_rows.parquet"
        if candidate.is_file():
            signed_candidates.append(candidate)

assert signed_candidates, "Full-coverage v0.15.1 signed rows not found"

SIGNED_SOURCE = signed_candidates[0]
signed = pd.read_parquet(SIGNED_SOURCE)

required_signed = {"H3_abs_slope","GT","delta_G","H3_signed_slope"}
assert required_signed.issubset(signed.columns), signed.columns.tolist()
assert len(signed) == 11596

print("Signed source:", SIGNED_SOURCE)
print("Signed rows:", len(signed))
print("V0.15.1 SIGNED SOURCE — PASS")


In [ ]:
TIE_EPS = 1e-12
signed_rows = []

for eps in THRESHOLDS:
    if eps < 0.002:
        signed_rows.append({
            "epsilon": float(eps),
            "status": "not_identifiable_from_v0151",
            "events": np.nan,
            "harmful_fraction_GT_positive": np.nan,
            "beneficial_fraction_GT_negative": np.nan,
            "tied_fraction": np.nan,
            "signed_slope_positive_fraction": np.nan,
            "delta_G_positive_fraction": np.nan,
        })
        continue

    g = signed[signed["H3_abs_slope"] > eps].copy()
    n = len(g)

    signed_rows.append({
        "epsilon": float(eps),
        "status": "identified",
        "events": int(n),
        "harmful_fraction_GT_positive": float((g["GT"] > TIE_EPS).mean()) if n else np.nan,
        "beneficial_fraction_GT_negative": float((g["GT"] < -TIE_EPS).mean()) if n else np.nan,
        "tied_fraction": float((np.abs(g["GT"]) <= TIE_EPS).mean()) if n else np.nan,
        "signed_slope_positive_fraction": float((g["H3_signed_slope"] > 0).mean()) if n else np.nan,
        "delta_G_positive_fraction": float((g["delta_G"] > 0).mean()) if n else np.nan,
    })

signed_sensitivity = pd.DataFrame(signed_rows)
display(signed_sensitivity)
signed_sensitivity.to_csv(
    OUT / "v016_signed_harm_threshold_sensitivity.csv",
    index=False,
)
print("SIGNED-HARM THRESHOLD SENSITIVITY — COMPLETE")


In [ ]:
primary_eps = 0.002

primary_global = config_corr_sensitivity[
    config_corr_sensitivity["epsilon"] == primary_eps
].iloc[0]

primary_resid = residual_corr[
    residual_corr["epsilon"] == primary_eps
].iloc[0]

primary_fit = regime_sensitivity[
    (regime_sensitivity["epsilon"] == primary_eps)
    & (regime_sensitivity["split"] == "fit")
].iloc[0]

primary_val = regime_sensitivity[
    (regime_sensitivity["epsilon"] == primary_eps)
    & (regime_sensitivity["split"] == "validation")
].iloc[0]

primary_signed = signed_sensitivity[
    signed_sensitivity["epsilon"] == primary_eps
].iloc[0]

summary = {
    "primary_epsilon": primary_eps,
    "fit_amplifying_fraction": float(primary_fit["amplifying_fraction"]),
    "validation_amplifying_fraction": float(primary_val["amplifying_fraction"]),
    "global_config_pearson": float(primary_global["pearson_r"]),
    "global_config_spearman": float(primary_global["spearman_rho"]),
    "alpha_centered_residual_pearson": float(
        primary_resid["within_alpha_centered_pearson_r"]
    ),
    "alpha_centered_residual_spearman": float(
        primary_resid["within_alpha_centered_spearman_rho"]
    ),
    "primary_signed_harmful_fraction": float(
        primary_signed["harmful_fraction_GT_positive"]
    ),
}

print(json.dumps(summary, indent=2))

(OUT / "v016_paper_facing_summary.json").write_text(
    json.dumps(summary, indent=2, sort_keys=True),
    encoding="utf-8",
)
print("PAPER-FACING SUMMARY — COMPLETE")


In [ ]:
report = {
    "status": "ARC_V016_THRESHOLD_ALPHA_ROBUSTNESS_COMPLETE",
    "source_v013_run": str(V013_RUN),
    "source_v013_protocol_sha256": sha256_file(PROTOCOL_PATH),
    "source_v013_validation_report_sha256": sha256_file(VALIDATION_REPORT_PATH),
    "source_v0151_signed_rows": str(SIGNED_SOURCE),
    "source_v0151_signed_rows_sha256": sha256_file(SIGNED_SOURCE),
    "thresholds": THRESHOLDS,
    "test_accessed": False,
    "interpretation_constraints": [
        "epsilon is an operational regime threshold; sensitivity evaluates robustness rather than a unique theoretical cutoff.",
        "within-alpha analyses address alpha confounding but do not prove causal transfer of each individual policy factor.",
        "signed harmfulness below epsilon=0.002 is not identifiable from the v0.15.1 replay."
    ],
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
}

REPORT_PATH = OUT / "v016_threshold_alpha_robustness_report.json"
REPORT_PATH.write_text(
    json.dumps(report, indent=2, sort_keys=True),
    encoding="utf-8",
)

report_sha = sha256_file(REPORT_PATH)
(OUT / "V016_REPORT_SHA256.txt").write_text(
    report_sha + "  " + REPORT_PATH.name + "\n",
    encoding="utf-8",
)

print("=" * 80)
print("ARC-v0.16 THRESHOLD / ALPHA ROBUSTNESS — PASS")
print("Output:", OUT)
print("Report SHA-256:", report_sha)
print("Test accessed:", False)
print("=" * 80)
